In [ ]:
# File: surface_mapping.ipynb
# Code: Claude Code
# Review: Ryoichi Ando (ryoichi.ando@zozo.com)
# License: Apache v2.0

In [ ]:
from frontend import App

# create an app for surface mapping demonstration
app = App.create("surface_mapping").clear_cache()

In [ ]:
import numpy as np

# create an icosphere surface mesh
surface_mesh = app.mesh.icosphere(r=1.0, subdiv_count=4)
orig_vert = surface_mesh[0].copy()
orig_tri = surface_mesh[1].copy()

print(f"Original surface: {len(orig_vert)} vertices, {len(orig_tri)} triangles")

# plot the original surface
app.plot.create().tri(orig_vert, orig_tri)

In [ ]:
# tetrahedralize (surface mapping is computed automatically)
tet_mesh = surface_mesh.tetrahedralize(edge_length_fac=0.1)

print(f"Tet mesh: {len(tet_mesh[0])} vertices, {len(tet_mesh[1])} surface tris, {len(tet_mesh[2])} tets")
print(f"Surface mapping exists: {tet_mesh.has_surface_mapping()}")

# plot the tet mesh
app.plot.create().tet(tet_mesh[0], tet_mesh[2])

In [ ]:
# interpolate with undeformed mesh - should recover original positions
interp_vert = tet_mesh.interpolate_surface()

print(f"Original shape: {orig_vert.shape}")
print(f"Interpolated shape: {interp_vert.shape}")

# compute error
errors = np.linalg.norm(interp_vert - orig_vert, axis=1)
print(f"Max error: {np.max(errors):.6e}")
print(f"Mean error: {np.mean(errors):.6e}")

In [ ]:
# test with deformation: scale by 2x
deformed_tet_vert = tet_mesh[0] * 2.0

# interpolate to get deformed original surface
interp_deformed = tet_mesh.interpolate_surface(deformed_tet_vert)

# expected: original vertices scaled by 2
expected = orig_vert * 2.0

errors_deformed = np.linalg.norm(interp_deformed - expected, axis=1)
print(f"Max error (scaled): {np.max(errors_deformed):.6e}")
print(f"Mean error (scaled): {np.mean(errors_deformed):.6e}")

# plot the interpolated deformed surface with ORIGINAL topology
app.plot.create().tri(interp_deformed, orig_tri)